<a href="https://colab.research.google.com/github/AlejoCNYT/aprMaqI_CEIA/blob/main/clase6/ejercicios/Solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Ejercicio 1: K-means sobre Iris
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix, adjusted_rand_score

# --- 1. Carga de datos (el CSV NO tiene header) ---
cols = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
df = pd.read_csv("iris.csv", header=___)            # PISTA: ¿qué valor indica "sin encabezado"? (None)
df.columns = cols
print(df.shape)
print(df["species"].value_counts())

X = df[cols[:4]]
y = df["species"]

# --- 2. Split 80/20 ANTES de cualquier ajuste (evitar data leakage) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=___,          # PISTA: 20% -> ¿qué valor decimal?
    stratify=___,           # PISTA: estratificar por la variable de clase
    random_state=42
)
# Guardar el 20% de test (requisito 1.1.2)
pd.concat([X_test, y_test], axis=1).to_csv("iris_test_20.csv", index=False)

# --- 3. Método del codo (solo con X_train) ---
inertias = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=___, n_init=10, random_state=42)   # PISTA: la variable del bucle
    km.fit(___)                                               # PISTA: ¿train o test?
    inertias.append(km.___)                                   # PISTA: atributo de sklearn con la inertia (termina en _)

plt.plot(list(k_range), inertias, marker="o")
plt.xlabel("K"); plt.ylabel("Inertia"); plt.title("Elbow method")
plt.xticks(list(k_range)); plt.grid(True); plt.show()

# --- 4. Modelo final con tu K óptimo ---
K_OPT = ___                 # PISTA: elígelo mirando el gráfico del codo
kmeans = KMeans(n_clusters=K_OPT, n_init=10, random_state=42).fit(X_train)

# Voto por mayoría: cluster -> especie más frecuente EN TRAIN
train_clusters = kmeans.___                       # PISTA: atributo con las etiquetas de cluster de train (labels_)
cluster_to_species = (
    pd.DataFrame({"cluster": train_clusters, "species": y_train.values})
      .groupby("cluster")["species"]
      .agg(lambda s: s.___().iloc[0])             # PISTA: método de pandas que da la moda (mode)
      .to_dict()
)
print("Mapeo cluster -> especie:", cluster_to_species)

# --- 5. Clasificar el 20% de test ---
test_clusters = kmeans.___(X_test)                # PISTA: método que asigna clusters a datos nuevos
y_pred = pd.Series(test_clusters).map(cluster_to_species)

# --- 6. Evaluación ---
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ARI     :", adjusted_rand_score(y_test, test_clusters))
print(pd.crosstab(pd.Series(test_clusters, name="cluster", index=y_test.index), y_test))